# Fixed-tree inference

Instead of inferring a local tree per window, as {class}`~ancestree.local_tree_inference.LocalTreeInference` does, the relationship between the ingroup and its outgroups can be assumed as one fixed tree, as is done in [`est-sfs`](https://sourceforge.net/projects/est-usfs/) ([Keightley and Jackson, 2018](https://doi.org/10.1534/genetics.118.301120)): the ingroup collapses to a polytomy under an inference root `I`, the outgroups join in a nested ladder of decreasing relatedness, and only the branch rates are fitted by maximum likelihood. {class}`~ancestree.trees.OutgroupLadderTree` represents that tree, and this guide walks {class}`~ancestree.inference.FixedTreeInference` end-to-end.

## Loading the tree sequence

The tree sequence (6 ingroup `i0..i5` + 2 outgroup `o0..o1`) supplies the truth for grading only.


In [1]:
import tskit

ts = tskit.load("quickstart.trees")  # ground truth for grading only
ingroup = ["i0", "i1", "i2", "i3", "i4", "i5"]
outgroup = ["o0", "o1"]


The inference reads a VCF exported from it, so only the genotype matrix carries information.


In [2]:
with open("quickstart.vcf", "w") as vcf:
    ts.write_vcf(vcf, contig_id="1", individual_names=ingroup + outgroup)


## The outgroup-ladder topology

{class}`~ancestree.trees.OutgroupLadderTree` represents that tree: the ingroup as the single node `I`, whose ancestral state is the quantity being inferred, and the branches `K0`, `K1`, … carrying the substitution rates that {meth}`FixedTreeInference.fit() <ancestree.inference.FixedTreeInference.fit>` estimates. Printing the tree shows that structure:


In [3]:
import ancestree as anc
tree = anc.OutgroupLadderTree(ingroup, outgroup)
print(tree.draw_text(show_branch_lengths=False))
print("params:", tree.param_names)


I
└── n_1
    ├── o0
    └── o1
params: ('K0', 'K1', 'K2')


## Running the inference

{meth}`Inference.from_fixed_tree() <ancestree.inference.Inference.from_fixed_tree>` builds a {class}`~ancestree.inference.FixedTreeInference` from the ingroup and outgroup names, and {meth}`FixedTreeInference.fit() <ancestree.inference.FixedTreeInference.fit>` ML-fits one rate per ladder branch against the polymorphic outgroup evidence and a monomorphic-site calibration. The ingroup enters through an ingroup weight, {class}`~ancestree.priors.KingmanIngroupWeight` by default or {class}`~ancestree.priors.AdaptiveIngroupWeight`, whose per-bin parameters are fitted in a second stage (see below), and the prior at the reporting node is a {class}`~ancestree.priors.StationaryPrior` passed as `prior=`, the model's stationary vector unless an empirical composition with counts is supplied. The fit also logs its MAP agreement with the majority-outgroup rule ({class}`~ancestree.inference.MajorityOutgroupInference`) per SFS bin, which disagrees wherever the ingroup is monomorphic, since the run reports at the ingroup MRCA.


In [4]:
vcf_inf = anc.Inference.from_fixed_tree(
    "quickstart.vcf", ingroup_samples=ingroup, outgroup_samples=outgroup,
    model=anc.JC69(), n_target_sites=int(ts.sequence_length),
)
vcf_inf.fit();


INFO:ancestree.CyVCF2Source: Genotypes read as ploidy 1; pass ploidy= to override
INFO:ancestree.CyVCF2Source: Reading variants from quickstart.vcf (8 haplotype samples)
INFO:ancestree.FixedTreeInference: Streaming fit: pass 1 (config histogram) complete; fitting 3 branch rate(s) by ML, then a second streaming pass emits posteriors.
INFO:ancestree.FixedTreeInference: 10/10 L-BFGS-B starts converged; best is start 2 (log L -70267.9684, range 0.0000); winning fit took 29 iterations
INFO:ancestree.FixedTreeInference: MLE branch rates: K0=2.0240e-03, K1=3.9990e-05, K2=6.0011e-05
INFO:ancestree.FixedTreeInference: Ingroup MRCA → outgroup divergences: o0=2.0640e-03, o1=2.0841e-03


In [5]:
res_vcf = list(vcf_inf.infer())


INFO:ancestree.FixedTreeInference: Inferring the ancestral allele on the fitted outgroup ladder, streaming the sites
FixedTreeInference: 224 sites [00:00, 28848.96 sites/s]
INFO:ancestree.FixedTreeInference: baseline (MajorityOutgroupInference) MAP agreement: 0.540 on 224 polymorphic sites; largest disagreement at SFS bin 0 (0.028); SFS bin 3 (1.000)
INFO:ancestree.FixedTreeInference: Focal node: reported at the ingroup_mrca


As in the {doc}`quickstart`, the result is graded on the sites polymorphic within the ingroup against the true allele at the ingroup MRCA.


In [6]:
keep = anc.PolymorphicSiteFilter(samples=ingroup)
truth = anc.Grade.truth_at_focal(ts, ingroup_samples=ingroup, outgroup_samples=outgroup)
anc.Grade(res_vcf, truth, filter=keep)


118 sites: MAP recovery 98.3%; mean Brier = 0.034

In [7]:
assert anc.Grade(res_vcf, truth, filter=keep).map_recovery > 0.95


:::{warning}
The number of mutational target sites matters. Most VCFs carry only polymorphic records, yet the fit must know how many monomorphic sites the genome contains to calibrate the branch rates against the fraction of polymorphism. Without that total the rates are unidentifiable. Supply {paramref}`n_target_sites <ancestree.inference.FixedTreeInference.n_target_sites>`, a {paramref}`base_composition <ancestree.inference.FixedTreeInference.base_composition>` ({class}`~ancestree.sites.BaseComposition`) with non-zero counts, or {paramref}`fit_required <ancestree.inference.FixedTreeInference.fit_required>``=False` to use the tree's branch lengths verbatim. If in doubt, err on a large count: it drives the fitted rates down, toward the maximum-parsimony limit, which costs little (see {doc}`arg_based_inference`).
:::

## Ingroup weight

With the ingroup collapsed at `I` there are no branches above `I` for the likelihood to score, so the outgroups carry nearly all of the signal and the ingroup enters only through its weight: the likelihood of the observed ingroup alleles given each candidate state at `I`, seeded on `I` inside the recursion. Under a neutral Kingman coalescent the more frequent allele is the more likely ancestral one. {class}`~ancestree.priors.NoIngroupWeight` switches the weight off, and two SFS-informed weights are available.

- {class}`~ancestree.priors.KingmanIngroupWeight`: closed-form. For a biallelic site with $n$ ingroup haplotypes of which $i$ carry the minor allele, the weight on the major allele being ancestral is $(n-i)/n$, the standard `est-sfs` weight.
- {class}`~ancestree.priors.AdaptiveIngroupWeight`: fits a per-SFS-bin parameter $\pi_i$ from the data in a second stage after the branch-rate fit, for an SFS that deviates from neutrality, and converges to Kingman in the neutral large-data limit.


## Specifying a fixed tree

When a published species tree is to be used as given, build the {class}`~ancestree.trees.OutgroupLadderTree` from its Newick string via {meth}`OutgroupLadderTree.from_newick() <ancestree.trees.OutgroupLadderTree.from_newick>` and pass `fit_required=False` to {class}`~ancestree.inference.FixedTreeInference` so the kernel runs against the supplied branch lengths verbatim.

In [8]:
newick = "(((i0,i1,i2,i3,i4,i5):0.01,o0:0.05):0.05,o1:0.10);"
tree_newick = anc.OutgroupLadderTree.from_newick(
    newick, ingroup_samples=ingroup, outgroup_samples=outgroup,
)
inf_newick = anc.FixedTreeInference(
    list(anc.TskitSource(ts)), anc.JC69(), anc.BaseComposition.no_counts(),
    tree=tree_newick, fit_required=False,
)
res_newick = list(inf_newick.infer())


INFO:ancestree.TskitSource: Reading variants from a tree sequence (224 sites, 8 samples)
INFO:ancestree.FixedTreeInference: Inferring the ancestral allele at 224 site(s) on the fitted outgroup ladder
FixedTreeInference: 100%|██████████| 224/224 [00:00<00:00, 39684.23 sites/s]
INFO:ancestree.FixedTreeInference: baseline (MajorityOutgroupInference) MAP agreement: 0.540 on 224 polymorphic sites; largest disagreement at SFS bin 0 (0.028); SFS bin 3 (1.000)
INFO:ancestree.FixedTreeInference: Focal node: reported at the ingroup_mrca


In [9]:
anc.Grade(res_newick, truth, filter=keep)


118 sites: MAP recovery 98.3%; mean Brier = 0.033

In [10]:
assert anc.Grade(res_newick, truth, filter=keep).map_recovery > 0.95


## Saving the output

{meth}`FixedTreeInference.to_vcf() <ancestree.inference.FixedTreeInference.to_vcf>` writes an annotated VCF with `AA`, `AA_prob` and `AA_post` `INFO` fields, each declared in the header (see {doc}`io`).


In [11]:
vcf_inf.to_vcf("snps.annotated.vcf.gz", posteriors=res_vcf);


INFO:ancestree.VCFWriter: Wrote 224 annotated sites to snps.annotated.vcf.gz


In [12]:
anc.Reader("snps.annotated.vcf.gz").head(3)


chrom   pos  alleles  AA       A       C       G       T
1       556  G/C      G   0.0000  0.0007  0.9993  0.0000
1       927  G/T      T   0.0000  0.0000  0.0000  1.0000
1      1031  T/A      T   0.0007  0.0000  0.0000  0.9993

In [13]:
from pathlib import Path

Path("snps.annotated.vcf.gz").unlink(missing_ok=True)
Path("quickstart.vcf").unlink(missing_ok=True)
